# 직접 해 보기 2 — 파이썬 코드로 따라 해 보기

AI가 내놓은 결과만 보고 그것이 어떻게 나왔는지 모르면, 그 결과를 믿을 근거도 없다. AI도 실수를 하므로 결과는 점검해야 하고, 점검하려면 결과를 만든 파이썬 코드를 읽을 줄 알아야 한다.

이 노트북은 이 장에서 다룬 계산과 그림을 파이썬으로 직접 해 본다. 일별 데이터와 시간별 데이터로 하루 한 행의 표를 만들고, 직선과 로지스틱 회귀로 내일 비가 올 확률을 설명하고, 계수를 오즈비와 확률의 말로 바꾸고, 모형이 말한 확률을 실제와 비교한다. 보고서의 표에 실린 숫자가 모두 여기서 나온다. 코드는 될 수 있는 한 단순하게 썼다.

1. 데이터 읽기 — 일별 데이터와 시간별 데이터
2. 하루 한 행 만들기 — 결과와 21시까지의 설명변수
3. 0과 1의 평균은 비율 — 구름양에 따른 내일 비의 비율
4. 직선으로 — 선형확률모형(표 2-2)
5. 로지스틱 회귀 — 보고서의 회귀 결과 표
6. 맑은 날과 흐린 날의 확률 — 같은 2hPa이 올리는 확률(표 4-2)
7. 오즈와 오즈비 — 어느 날에나 같은 배수
8. 평균 한계효과 — 모든 날의 평균(보고서의 확률 변화)
9. 모형이 말한 확률과 실제 — 표 6-1
10. 측정 시점 — 오늘비를 하루 전체로 바꾸면

## 준비 — 패키지와 글꼴

`import`로 쓸 패키지를 불러온다. 다른 장과 같은 네 가지다. 로지스틱 회귀도 statsmodels에 들어 있다.

matplotlib는 기본 글꼴에 한글이 없어서 그림의 한글이 네모로 깨진다. 그래서 한글 글꼴을 지정한다. 윈도우는 맑은 고딕(`Malgun Gothic`), 맥은 애플고딕(`AppleGothic`)을 쓴다. Colab에는 한글 글꼴이 없으므로 새 셀에서 `!apt-get -qq install fonts-nanum`을 실행하고, 런타임을 다시 시작한 뒤 `NanumGothic`을 지정한다.

In [ ]:
import numpy as np                       # 수치 계산
import pandas as pd                      # 표(데이터프레임) 다루기
import statsmodels.formula.api as smf    # 회귀분석
import matplotlib.pyplot as plt          # 그림

# 그림의 한글 글꼴
# 맥은 'AppleGothic', Colab은 'NanumGothic'(설치 필요)
plt.rcParams['font.family'] = 'Malgun Gothic'   # 윈도우
plt.rcParams['axes.unicode_minus'] = False      # 음수 부호(-) 깨짐 방지

## 1. 데이터 읽기

데이터는 기상청 종관기상관측(ASOS) 서울 지점의 2016 ~ 2025년 기록이다. 두 파일을 쓴다. 책 사이트의 이 장 첫 페이지에서 내려받을 수 있고, 다른 지점이나 기간은 기상청 기상자료개방포털(data.kma.go.kr)에서 받는다.

- **일별 데이터**(`DAILY`): 한 행이 하루다. 여기서는 그날의 강수량(`일강수량(mm)`)만 쓴다.
- **시간별 데이터**(`HOURLY`): 한 행이 한 시간이다. 그 시각의 습도, 해면기압, 전운량, 강수량을 쓴다. 10년이면 8만 행이 넘는다.

- 내 컴퓨터의 주피터: 두 CSV를 이 노트북 파일과 같은 폴더에 두고, 아래 코드에 파일 이름을 그대로 적는다.
- Colab: 왼쪽의 폴더 아이콘을 누르고 업로드 버튼으로 두 CSV를 올린 뒤 `'/content/파일이름.csv'`로 적는다.

`encoding='utf-8-sig'`는 파일 맨 앞에 붙은 보이지 않는 표시(BOM)를 떼어 내는 옵션이다. 이것이 없으면 첫 열의 이름이 `지점`이 아니라 `﻿지점`으로 읽힌다.

In [ ]:
DAILY = 'asos-seoul-daily-2016-2025.csv'     # 일별 데이터, 자기 파일 이름으로 바꾼다
HOURLY = 'asos-seoul-hourly-2016-2025.csv'   # 시간별 데이터

dd = pd.read_csv(DAILY, encoding='utf-8-sig')
hh = pd.read_csv(HOURLY, encoding='utf-8-sig')
print('일별 데이터', len(dd), '행, 시간별 데이터', len(hh), '행')
hh.head()

## 2. 하루 한 행 만들기

1절의 "데이터"에서 한 일이다. 한 행을 하루로 하는 표 `d`를 만든다.

**결과: 내일 비가 왔나.** 일별 데이터의 일강수량이 0.1mm 이상이면 그날 비가 온 것으로 본다(1). 기상청 일자료는 비가 오지 않은 날의 일강수량을 0이 아니라 **빈칸**으로 둔다. 그래서 `fillna(0)`으로 빈칸을 0으로 채운다. 빈칸을 빠진 값으로 여겨 버리면 비가 온 날만 잔뜩 남는다. `shift(-1)`은 한 행씩 위로 당겨, 오늘 행에 다음 날의 값을 놓는다.

**설명변수: 오늘 21시에 이미 알 수 있는 값만.** 시간별 데이터에서 만든다.

- 저녁 습도(`hum`): 21시의 상대습도
- 기압 변화(`chg`): 21시 해면기압 − 9시 해면기압. 음수면 낮 동안 기압이 내린 것이다
- 저녁 구름양(`cloud`): 21시의 전운량(0 ~ 10)
- 오늘비(`rain`): 1시부터 21시 사이에 강수량이 기록된 시간이 하나라도 있으면 1. 시간별 데이터의 1시 기록은 0시 ~ 1시의 강수이므로, 오늘 0시부터 21시까지다
- 여름(`summer`): 6 ~ 8월이면 1

`at(21)`은 시간별 데이터에서 21시 행만 골라 날짜를 행 이름으로 삼은 표다. 날짜가 같은 행끼리 짝이 맞춰져 `d`에 들어간다. 값이 빠진 날을 빼면 3,651일이 남아야 한다.

In [ ]:
# 시간별 데이터: 날짜와 시각을 나눈다
hh['t'] = pd.to_datetime(hh['일시'])
hh['날짜'] = hh['t'].dt.normalize()     # 2016-01-01 21:00 → 2016-01-01
hh['시'] = hh['t'].dt.hour              # 2016-01-01 21:00 → 21

def at(hour):                           # 그 시각의 행만, 날짜를 행 이름으로
    return hh[hh['시'] == hour].set_index('날짜')

# 하루 한 행의 표
d = pd.DataFrame(index=pd.to_datetime(dd['일시']))
비 = (dd['일강수량(mm)'].fillna(0) >= 0.1).astype(int).values   # 빈칸 = 비가 오지 않은 날
d['tomorrow'] = pd.Series(비, index=d.index).shift(-1)          # 다음 날의 값을 오늘 행에

d['hum'] = at(21)['습도(%)']
d['chg'] = at(21)['해면기압(hPa)'] - at(9)['해면기압(hPa)']
d['cloud'] = at(21)['전운량(10분위)']
낮 = hh[(hh['시'] >= 1) & (hh['시'] <= 21)]
d['rain'] = (낮['강수량(mm)'].fillna(0) > 0).groupby(낮['날짜']).any().astype(int)
d['summer'] = d.index.month.isin([6, 7, 8]).astype(int)

print('모든 날', len(d), '일')
d = d.dropna()                          # 값이 빠진 날을 뺀다
print('분석한 날', len(d), '일, 다음 날 비가 온 날', int(d['tomorrow'].sum()), '일')
d.head()

## 3. 0과 1의 평균은 비율

2절의 "0과 1의 평균은 비율이다"에서 한 일이다. 결과가 0과 1이면 그 평균이 곧 1의 비율이다. 3,651일의 `tomorrow`를 평균하면 다음 날 비가 온 날의 비율, 29.5%가 나온다.

`groupby('cloud')`로 21시 구름양이 같은 날끼리 묶어 평균하면, 구름양마다 다음 날 비가 온 비율이 나온다. 1절에서 본 숫자다. 구름이 한 점도 없던 날(0)은 11%, 하늘을 다 덮은 날(10)은 72%다. 조건이 같은 날들의 평균, 즉 비율이 그 조건에서의 확률이다.

In [ ]:
print('다음 날 비가 온 비율', round(d['tomorrow'].mean() * 100, 1), '%')

by_cloud = d.groupby('cloud')['tomorrow'].agg(['size', 'mean'])
by_cloud.columns = ['일수', '내일 비의 비율']
by_cloud['내일 비의 비율'] = (by_cloud['내일 비의 비율'] * 100).round(1)
by_cloud

## 4. 직선으로 — 선형확률모형

2절에서 한 일이다. 0과 1인 결과에 지금까지와 같은 회귀식(`smf.ols`)을 그대로 쓴다. 회귀식은 다른 장과 같은 방식으로 적는다. `~` 왼쪽이 종속변수, 오른쪽이 설명변수다.

`cov_type='HC1'`은 표준오차를 이분산 강건 표준오차로 계산하라는 옵션이다(2절의 참고 상자). 계수는 그대로이고 표준오차만 달라진다.

계수는 퍼센트포인트로 바로 해석한다. 기압 변화의 계수 −0.0256은 기압이 1hPa 오른 날 다음 날 비가 올 확률이 2.6%포인트 낮다는 뜻이다. `fittedvalues`는 회귀식이 날마다 계산한 값, 즉 내일비 확률이다. 이 값이 0보다 낮은 날이 138일이다.

In [ ]:
F = 'tomorrow ~ hum + chg + cloud + rain + summer'   # 보고서의 회귀식

lpm = smf.ols(F, d).fit(cov_type='HC1')
print(pd.DataFrame({'계수': lpm.params, '표준오차': lpm.bse}).round(4))

p_line = lpm.fittedvalues                             # 날마다의 내일비 확률(직선)
print()
print('0%보다 낮은 날', (p_line < 0).sum(), '일, 100%보다 높은 날', (p_line > 1).sum(), '일')

## 5. 로지스틱 회귀

3절에서 한 일이다. `smf.ols`를 `smf.logit`으로 바꾸기만 하면 된다. 계수는 최대가능도 추정으로 계산된다. 컴퓨터가 계수를 조금씩 바꿔 가며 가능도가 더 커지지 않는 곳을 찾으므로, `disp=0`으로 그 과정의 메시지를 끈다.

표의 계수, 표준오차, $p$값이 보고서의 회귀 결과 표와 같다. 오즈비는 7절, 확률 변화는 8절에서 계산한다.

계수는 점수를 만든다. 점수는 상수에 변수마다 계수를 곱해 더한 값이고, 로지스틱 곡선 $1 / (1 + e^{-점수})$에 넣으면 확률이 된다. `predict`가 이 계산을 날마다 해 준다. 모든 날의 확률이 0%와 100% 사이에 있다.

In [ ]:
lg = smf.logit(F, d).fit(disp=0)
print(pd.DataFrame({'계수': lg.params, '표준오차': lg.bse, 'p값': lg.pvalues}).round(3))

d['p'] = lg.predict(d)                                # 날마다의 내일비 확률(곡선)
print()
print('확률의 범위', round(d['p'].min() * 100, 1), '% ~', round(d['p'].max() * 100, 1), '%')

## 6. 맑은 날과 흐린 날의 확률

3절과 4절에서 한 일이다. 보고서가 고른 두 날을 표로 만들어 `predict`에 넣는다. 맑고 건조한 날은 21시 습도 50%, 구름양 2이고, 흐리고 습한 날은 습도 85%, 구름양 8이다. 둘 다 오늘 비가 오지 않았고 여름이 아니다. 기압 변화만 0, −2, −4hPa로 바꿔 가며 확률을 계산한다.

맑은 날은 11.5% → 15.1%, 흐린 날은 36.0% → 43.4%다. 같은 2hPa이 맑은 날에는 3.5%포인트, 흐린 날에는 7.4%포인트를 올린다. 흐린 날 기압이 4hPa 내리면 51%로, 보고서 요약의 첫 줄이다.

그림은 보고서 그림의 왼쪽 패널을 단순하게 다시 그린 것이다. `np.linspace(-12, 6, 200)`은 −12부터 6까지를 200개로 고르게 나눈 값이다.

In [ ]:
CLEAR = dict(hum=50, cloud=2, rain=0, summer=0)       # 맑고 건조한 날
CLOUDY = dict(hum=85, cloud=8, rain=0, summer=0)      # 흐리고 습한 날

def prob(day, chg):                                   # 그날 기압 변화가 chg일 때의 확률
    x = pd.DataFrame([{**day, 'chg': c} for c in chg])
    return np.asarray(lg.predict(x))

for name, day in [('맑은 날', CLEAR), ('흐린 날', CLOUDY)]:
    p = prob(day, [0, -2, -4]) * 100
    print(name, '기압 변화 0, −2, −4hPa:', p.round(1), '  2hPa 더 내리면', round(p[1] - p[0], 1), '%포인트')

xs = np.linspace(-12, 6, 200)
plt.figure(figsize=(6, 3.6))
plt.plot(xs, prob(CLOUDY, xs) * 100, label='흐리고 습한 날')
plt.plot(xs, prob(CLEAR, xs) * 100, label='맑고 건조한 날')
plt.xlabel('기압 변화 (hPa, 21시 - 9시)')
plt.ylabel('내일 비가 올 확률 (%)')
plt.ylim(0, 100)
plt.legend()
plt.show()

## 7. 오즈와 오즈비

4절에서 한 일이다. 오즈는 비가 올 확률을 비가 오지 않을 확률로 나눈 값이다. 오즈비는 계수 $b$로 $e^{b}$를 계산한 것이다. `np.exp`가 $e$의 거듭제곱을 계산한다. 기압 변화의 오즈비 0.86은 기압이 1hPa 오르면 오즈가 0.86배가 된다는 뜻이고, 1 ÷ 0.86 ≈ 1.17은 1hPa 내리면 1.17배가 된다는 뜻이다.

오즈비가 어느 날에나 같은지 확인한다. 6절의 두 날에서 기압이 2hPa 더 내릴 때 오즈가 몇 배가 되는지 계산하면, 확률의 차이는 두 배 넘게 다른데 오즈의 배수는 둘 다 1.36이다(표 4-2).

In [ ]:
print('오즈비'); print(np.exp(lg.params).round(2))
print()
print('기압이 1hPa 내리면 오즈가', round(1 / np.exp(lg.params['chg']), 2), '배')
print()

def odds(p):
    return p / (1 - p)

for name, day in [('맑은 날', CLEAR), ('흐린 날', CLOUDY)]:
    p0, p2 = prob(day, [0, -2])
    print(name, ' 오즈', round(odds(p0), 3), '→', round(odds(p2), 3),
          ' 오즈의 배수', round(odds(p2) / odds(p0), 2), ' 확률의 차이', round((p2 - p0) * 100, 1), '%포인트')

## 8. 평균 한계효과

4절의 "확률의 차이 — 두 가지 방법"에서 둘째 방법이다. 확률의 차이는 날마다 다르므로 모든 날의 평균을 낸다. `get_margeff(at='overall')`이 날마다 "변수가 조금 더 클 때 확률이 얼마나 오르나"를 계산해 평균한다. 기압 변화는 −2.6%포인트로, 보고서 표의 확률 변화다.

같은 계산을 직접 해 볼 수도 있다. 로지스틱 곡선의 기울기는 그날의 확률이 $p$일 때 $p(1-p)$이고, 여기에 계수를 곱하면 그날의 한계효과다. 확률이 50%인 날에 기울기가 가장 크다. 날마다 계산해 평균하면 같은 −2.6%포인트가 나온다.

마지막 셀은 4절의 "오즈비는 확률의 배수가 아니다"에서 한 계산이다. 모든 날을 여름이라고 했을 때와 여름이 아니라고 했을 때의 확률을 각각 평균하면 37.0%와 26.5%다. 확률로는 1.40배로, 여름의 오즈비 1.83과 꽤 다르다.

In [ ]:
ame = lg.get_margeff(at='overall').summary_frame()
print('평균 한계효과(%포인트)'); print((ame['dy/dx'] * 100).round(2))

# 직접 계산: 날마다 계수 × p(1 − p)를 계산해 평균
b = lg.params['chg']
print()
print('직접 계산한 기압 변화의 평균 한계효과', round((b * d['p'] * (1 - d['p'])).mean() * 100, 2), '%포인트')

In [ ]:
여름 = d.assign(summer=1)                             # 모든 날을 여름이라고 하면
여름아님 = d.assign(summer=0)                         # 모든 날을 여름이 아니라고 하면
a, c = lg.predict(여름).mean(), lg.predict(여름아님).mean()
print('여름', round(a * 100, 1), '%, 여름 아님', round(c * 100, 1), '%, 확률로는', round(a / c, 2), '배')
print('여름의 오즈비', round(np.exp(lg.params['summer']), 2))

## 9. 모형이 말한 확률과 실제

6절에서 한 일이다. 날마다의 확률(`p`)을 10%포인트씩 구간으로 나누고, 구간마다 일수, 모형이 말한 확률의 평균, 다음 날 실제로 비가 온 비율을 계산한다. `pd.cut`이 값을 구간으로 나눈다. `np.linspace(0, 1, 11)`은 0, 0.1, …, 1의 열한 경계다.

20 ~ 30% 구간을 보면 461일, 모형이 말한 확률의 평균 24.8%, 실제 23.4%다. 60%를 넘는 구간에서는 실제 비율이 모형보다 높다. 그림의 점이 점선(대각선) 위에 있으면 모형이 말한 대로 비가 온 것이다.

In [ ]:
구간 = pd.cut(d['p'], np.linspace(0, 1, 11))
cal = d.groupby(구간, observed=True).agg(일수=('tomorrow', 'size'),
                                          모형의_확률=('p', 'mean'),
                                          실제_비율=('tomorrow', 'mean'))
cal[['모형의_확률', '실제_비율']] = (cal[['모형의_확률', '실제_비율']] * 100).round(1)
print(cal)

plt.figure(figsize=(4, 4))
plt.plot([0, 100], [0, 100], ls='--', color='gray')   # 말한 대로라면 이 선 위
plt.scatter(cal['모형의_확률'], cal['실제_비율'], s=cal['일수'] / 5)   # 점의 크기 = 일수
plt.xlabel('모형이 말한 확률 (%)')
plt.ylabel('다음 날 실제로 비가 온 비율 (%)')
plt.show()

## 10. 측정 시점 — 오늘비를 하루 전체로 바꾸면

6절의 "오늘 밤에 알 수 있는 것만 쓴다"에서 한 일이다. 오늘비를 21시까지가 아니라 일별 데이터의 일강수량(0 ~ 24시)으로 만들면 계수가 0.281에서 0.472로 커진다. 21시에서 자정 사이에 내린 비, 즉 오늘 21시에는 아직 알 수 없는 정보가 섞여 들어갔기 때문이다.

`d['rain24']`는 2절에서 결과를 만들 때 쓴 `비`를 하루 늦추지 않고 그대로 붙인 것이다. 2절에서 값이 빠진 날을 뺐으므로, 날짜로 짝을 맞춰 붙인다.

In [ ]:
오늘비24 = pd.Series(비, index=pd.to_datetime(dd['일시']))      # 오늘 0 ~ 24시의 비
d['rain24'] = 오늘비24.reindex(d.index)

lg24 = smf.logit('tomorrow ~ hum + chg + cloud + rain24 + summer', d).fit(disp=0)
print('오늘비(21시까지)의 계수', round(lg.params['rain'], 3))
print('오늘비(하루 전체)의 계수', round(lg24.params['rain24'], 3))

## 정리

이 노트북의 코드는 이 장에서 본 숫자를 직접 만들어 냈다. 보고서의 숫자는 세 가지에서 나온다. 하루 한 행의 표를 만드는 2절, 로지스틱 회귀 한 줄(5절의 `smf.logit`), 그리고 그 결과를 확률의 말로 바꾸는 `predict`와 `get_margeff`(6 ~ 9절)다. 이 가운데 AI가 가장 틀리기 쉬운 곳은 2절이다. 일강수량의 빈칸을 0으로 채웠는지, 설명변수를 21시까지의 값으로 만들었는지에 따라 숫자가 달라진다(10절).

자기 데이터로 분석하려면 1절의 두 파일 이름을 바꾼다. 기상자료개방포털에서 받은 파일이라면 열 이름이 같으므로 나머지는 그대로 돌아간다. 다른 지역은 계절이 다를 수 있으니, 2절의 여름(6 ~ 8월)이 그 지역에도 맞는지 생각해 본다.

AI가 만든 보고서를 받았다면, 이 코드로 같은 숫자가 나오는지 확인해 보는 것이 가장 확실한 점검이다.